In [1]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
#from langchain.text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

In [3]:
load_dotenv()

PROJECT_ROOT = Path("..").resolve()
KNOWLEDGE_BASE_DIR = PROJECT_ROOT / "knowledge_base"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CHROMA_DIR = ARTIFACTS_DIR / "chroma_db"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("KNOWLEDGE_BASE_DIR:", KNOWLEDGE_BASE_DIR)
print("CHROMA_DIR:", CHROMA_DIR)

PROJECT_ROOT: /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI
KNOWLEDGE_BASE_DIR: /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/knowledge_base
CHROMA_DIR: /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/artifacts/chroma_db


In [4]:
openai_key = os.getenv("OPENAI_API_KEY")

if not openai_key:
    raise ValueError("OPENAI_API_KEY not found. Add it to your environment or .env file.")

print("OPENAI_API_KEY found.")

OPENAI_API_KEY found.


In [5]:
loader = DirectoryLoader(
    str(KNOWLEDGE_BASE_DIR),
    glob="**/*.md",
    loader_cls=TextLoader
)

docs = loader.load()

print("Number of loaded docs:", len(docs))
for i, doc in enumerate(docs, start=1):
    print(f"{i}.", doc.metadata.get("source"))

Number of loaded docs: 4
1. /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/knowledge_base/improvement_tips.md
2. /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/knowledge_base/faq.md
3. /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/knowledge_base/feature_meaning.md
4. /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/knowledge_base/approval_rules.md


In [6]:
for i, doc in enumerate(docs[:2], start=1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:1000])


--- Document 1 ---
# Improvement Tips

- Improve your **cibil_score** before reapplying, since credit score is the strongest factor in the model.
- Review whether the chosen **loan_term** is making the application less favorable.
- Request a more reasonable **loan_amount** relative to your financial profile.
- Strengthen proof of **annual income** if possible.
- Improve the declared financial strength of the application through stronger **bank**, **residential**, or **commercial asset** support where applicable.
- Make sure all submitted details are accurate and complete.
- If financial burden is high, consider reducing risk factors before reapplying.

--- Document 2 ---
# FAQ

## Why was my loan application rejected?
Common reasons may include a weaker credit score, an unfavorable loan term, lower income strength relative to the requested loan, or weaker overall financial support from declared assets.

## Why was my loan application approved?
Applications are more likely to be approv

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(docs)

print("Number of chunks:", len(split_docs))
print("\nSample chunk:\n")
print(split_docs[0].page_content)

Number of chunks: 14

Sample chunk:

# Improvement Tips


In [8]:
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR)
)

print("Chroma vector store created successfully.")

Chroma vector store created successfully.


In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever ready.")

Retriever ready.


In [16]:
test_query = "Why was the loan application rejected?"
retrieved_docs = retriever.invoke(test_query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Doc {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content)


--- Retrieved Doc 1 ---
Source: /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/knowledge_base/faq.md
# FAQ

## Why was my loan application rejected?
Common reasons may include a weaker credit score, an unfavorable loan term, lower income strength relative to the requested loan, or weaker overall financial support from declared assets.

## Why was my loan application approved?
Applications are more likely to be approved when the applicant shows strong creditworthiness, stronger income, and a healthier financial profile supported by assets.

--- Retrieved Doc 2 ---
Source: /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/knowledge_base/improvement_tips.md
- Improve your **cibil_score** before reapplying, since credit score is the strongest factor in the model.
- Review whether the chosen **loan_term** is making the application less favorable.
- Request a more reasonable **loan_amount** relative to your financial profile.
- Strengthen proof of **annual income** if p

### example inference input data



In [11]:
input_data = {
    "no_of_dependents": 2,
    "income_annum": 850000,
    "loan_amount": 1200000,
    "loan_term": 18,
    "cibil_score": 812,
    "residential_assets_value": 1800000,
    "commercial_assets_value": 500000,
    "luxury_assets_value": 200000,
    "bank_asset_value": 300000,
    "education": "Graduate",
    "self_employed": "No"
}

input_data

{'no_of_dependents': 2,
 'income_annum': 850000,
 'loan_amount': 1200000,
 'loan_term': 18,
 'cibil_score': 812,
 'residential_assets_value': 1800000,
 'commercial_assets_value': 500000,
 'luxury_assets_value': 200000,
 'bank_asset_value': 300000,
 'education': 'Graduate',
 'self_employed': 'No'}

In [12]:
prediction_result = {
    "prediction": "Approved",
    "probability": 0.93
}

prediction_result

{'prediction': 'Approved', 'probability': 0.93}

In [13]:
local_explanation = {
    "top_positive": {
        "num__cibil_score": 2.45,
        "num__residential_assets_value": 1.34,
        "num__commercial_assets_value": 1.05,
        "num__bank_asset_value": 0.88,
        "num__loan_amount": 0.30
    },
    "top_negative": {
        "num__loan_term": -0.60,
        "num__income_annum": -0.35,
        "num__luxury_assets_value": -0.28,
        "cat__education_ Graduate": -0.10,
        "num__no_of_dependents": -0.08
    }
}

local_explanation

{'top_positive': {'num__cibil_score': 2.45,
  'num__residential_assets_value': 1.34,
  'num__commercial_assets_value': 1.05,
  'num__bank_asset_value': 0.88,
  'num__loan_amount': 0.3},
 'top_negative': {'num__loan_term': -0.6,
  'num__income_annum': -0.35,
  'num__luxury_assets_value': -0.28,
  'cat__education_ Graduate': -0.1,
  'num__no_of_dependents': -0.08}}

In [14]:
def format_retrieved_context(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs, start=1):
        source = os.path.basename(doc.metadata.get("source", "unknown"))
        content = doc.page_content.strip()
        parts.append(f"[Document {i}: {source}]\n{content}")
    return "\n\n".join(parts)

In [15]:
def format_dict_pretty(data):
    return json.dumps(data, indent=2)

In [17]:
prompt = ChatPromptTemplate.from_template("""
You are a loan approval assistant.

Use the following sources of information:
1. Applicant input data
2. Model prediction result
3. Local explanation from the model
4. Retrieved knowledge base context

Applicant input data:
{input_data}

Prediction result:
{prediction_result}

Local explanation:
{local_explanation}

Retrieved rule context:
{retrieved_context}

User question:
{question}

Instructions:
- Use the applicant input values when relevant.
- Clearly mention the final model decision.
- Use the local explanation to describe what pushed the decision toward approval or rejection.
- Use the retrieved rule context as grounding.
- Do not invent unsupported reasons.
- Keep the answer practical and human-readable.
- If giving advice, make it consistent with the retrieved context and model explanation.
""")

In [18]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM ready.")

LLM ready.


In [19]:
question = "Why was this loan approved?"

retrieved_docs = retriever.invoke(question)
retrieved_context = format_retrieved_context(retrieved_docs)

messages = prompt.format_messages(
    input_data=format_dict_pretty(input_data),
    prediction_result=format_dict_pretty(prediction_result),
    local_explanation=format_dict_pretty(local_explanation),
    retrieved_context=retrieved_context,
    question=question
)

response = llm.invoke(messages)
print(response.content)

The loan application was approved with a high confidence of 93%. 

This positive decision is mainly driven by the applicant’s strong credit score of 812, which is the most influential factor in the model and contributed the most to approval. Additionally, the applicant’s substantial residential assets (valued at 1,800,000), commercial assets (500,000), and bank assets (300,000) further strengthened the financial profile, supporting the loan request of 1,200,000.

While some factors slightly reduced the approval likelihood—such as the relatively long loan term of 18 years, the number of dependents (2), and the luxury assets value—the overall financial strength and creditworthiness outweighed these negatives.

According to the approval rules and FAQs, strong creditworthiness, solid asset backing, and a reasonable income relative to the loan amount are key to approval, all of which are present in this case.

In summary, the loan was approved because the applicant demonstrates a healthy fi

In [20]:
def answer_loan_question(
    question: str,
    input_data: dict,
    prediction_result: dict,
    local_explanation: dict,
    retriever,
    llm,
    top_k: int = 3
):
    retrieved_docs = retriever.invoke(question)
    retrieved_context = format_retrieved_context(retrieved_docs)

    messages = prompt.format_messages(
        input_data=format_dict_pretty(input_data),
        prediction_result=format_dict_pretty(prediction_result),
        local_explanation=format_dict_pretty(local_explanation),
        retrieved_context=retrieved_context,
        question=question
    )

    response = llm.invoke(messages)

    return {
        "question": question,
        "answer": response.content,
        "retrieved_docs": [
            {
                "source": doc.metadata.get("source"),
                "content": doc.page_content
            }
            for doc in retrieved_docs[:top_k]
        ]
    }

In [21]:
questions = [
    "Why was this loan approved?",
    "What factors helped this application?",
    "What factors weakened this application?",
    "What should improve before reapplying?",
    "Does credit score matter a lot?",
    "Summarize this application for an analyst."
]

for q in questions:
    result = answer_loan_question(
        question=q,
        input_data=input_data,
        prediction_result=prediction_result,
        local_explanation=local_explanation,
        retriever=retriever,
        llm=llm
    )

    print("\n" + "=" * 80)
    print("QUESTION:", result["question"])
    print("-" * 80)
    print(result["answer"])


QUESTION: Why was this loan approved?
--------------------------------------------------------------------------------
The loan application was approved with a high confidence of 93%. 

This positive decision is mainly driven by the applicant’s strong creditworthiness and financial profile. Specifically, the high CIBIL score of 812 had the largest positive impact on approval, as credit score is the strongest factor in the model. Additionally, the applicant’s substantial residential assets (valued at 1,800,000), commercial assets (500,000), and bank assets (300,000) further strengthened the application. These asset values indicate a healthy financial backing, which supports the loan request.

On the other hand, some factors slightly reduced the approval likelihood, such as the relatively long loan term of 18 months, the applicant’s income level, the presence of luxury assets, and having two dependents. However, these negative influences were outweighed by the strong positive factors.



In [22]:
result = answer_loan_question(
    question="What should improve before reapplying?",
    input_data=input_data,
    prediction_result=prediction_result,
    local_explanation=local_explanation,
    retriever=retriever,
    llm=llm
)

print("Answer:\n")
print(result["answer"])

print("\nRetrieved Docs:\n")
for i, doc in enumerate(result["retrieved_docs"], start=1):
    print(f"\n--- Doc {i} ---")
    print("Source:", doc["source"])
    print(doc["content"])

Answer:

Your loan application has been **approved** with a high confidence of 93%. The strongest positive factors contributing to this approval are your excellent **CIBIL score (812)**, substantial **residential assets value (₹1,800,000)**, good **commercial assets (₹500,000)**, and solid **bank assets (₹300,000)**. These financial strengths significantly outweighed the negative influences.

However, some factors slightly reduced the favorability of your application:
- The relatively long **loan term (18 months)** had a negative impact.
- Your **annual income (₹850,000)**, while decent, was less influential than your assets.
- The presence of **luxury assets (₹200,000)** and your education level (Graduate) also had minor negative effects.
- Having **2 dependents** slightly lowered the score as well.

**Improvement suggestions before reapplying** (if you wish to strengthen your application further or if you plan to apply for a larger loan):
- Consider shortening the **loan term** to re

In [23]:
sample_output_path = ARTIFACTS_DIR / "phase5_agent_sample_output.json"

sample_result = answer_loan_question(
    question="Why was this loan approved?",
    input_data=input_data,
    prediction_result=prediction_result,
    local_explanation=local_explanation,
    retriever=retriever,
    llm=llm
)

with open(sample_output_path, "w") as f:
    json.dump(sample_result, f, indent=2)

print("Saved sample agent output to:", sample_output_path)

Saved sample agent output to: /Users/pranavsrinivasvenkatesh/Projects/ML with Agentic AI/artifacts/phase5_agent_sample_output.json


In [24]:
def extract_local_reason_summary(contrib_df, row_idx, top_n=5):
    row_contrib = contrib_df.loc[row_idx].drop("bias").sort_values()
    top_negative = row_contrib.head(top_n)
    top_positive = row_contrib.tail(top_n).sort_values(ascending=False)

    return {
        "top_positive": top_positive.to_dict(),
        "top_negative": top_negative.to_dict()
    }

In [25]:
row_idx = 0
local_explanation = extract_local_reason_summary(contrib_df, row_idx=row_idx, top_n=5)

NameError: name 'contrib_df' is not defined

In [ ]:
pred_prob = model.predict_proba(X_processed_df.iloc[[0]])[:, 1][0]
pred_label = "Approved" if pred_prob >= 0.5 else "Rejected"

prediction_result = {
    "prediction": pred_label,
    "probability": float(pred_prob)
}